## Setup



**Note**: TrustworthyAgent requires a data directory for storing trustworthiness scores:

```bash

mkdir -p data/trustworthy_react_openai

```


In [1]:
# Create required directory for TrustworthyAgent

import os

os.makedirs('data/trustworthy_react_openai', exist_ok=True)

print('✅ Created data directory for TrustworthyAgent')


✅ Created data directory for TrustworthyAgent


# Google Sheets Manager Agent Tutorial



This notebook demonstrates building a TrustworthyAgent for managing Google Sheets.



The SheetManagerAgent uses TrustworthyAgent as its base class, providing:

- **Trustworthiness scores** for each action

- **Multiple reasoning modes**: react, act, planact, planreact

- **Cleanlab TLM integration** for monitoring agent reliability


## Prerequisites
1. Create a Google Cloud Project.
2. Enable the Google Sheets API.
3. Create a Service Account and download the credentials JSON.
4. Set the `GOOGLE_SHEETS_CREDENTIALS` environment variable.
5. Share your spreadsheet with the service account email.

Refer to https://docs.gspread.org/en/latest/oauth2.html#for-bots-using-service-account for detailed setup instructions.

In [2]:
import os
import sys
import importlib
from dotenv import load_dotenv

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
    print(f"🔧 Added project root to sys.path: {PROJECT_ROOT}")
else:
    print(f"📁 Project root already on sys.path: {PROJECT_ROOT}")

_agentlite_modules = [name for name in list(sys.modules.keys()) if name == "agentlite" or name.startswith("agentlite.")]

if _agentlite_modules:
    for name in _agentlite_modules:
        del sys.modules[name]
    importlib.invalidate_caches()
    print(f"🧼 Cleared cached agentlite modules: {len(_agentlite_modules)} removed")

import agentlite
print(f"📦 Using agentlite package from: {agentlite.__file__}")
from agentlite.commons import TaskPackage
from agentlite.llm.LLMConfig import LLMConfig
from agentlite.llm.agent_llms import get_llm_backend
from sheet_tool_operation import get_gspread_client
from sheet_tool_operation.sheet_agent import SheetManagerAgent


🔧 Added project root to sys.path: /Users/ymmtny/Documents/GitHub/AgentLiteTLM
📦 Using agentlite package from: /Users/ymmtny/Documents/GitHub/AgentLiteTLM/agentlite/__init__.py


In [3]:
def verify_environment():
    """Verify that we're running in the correct conda environment."""
    conda_env = os.environ.get("CONDA_DEFAULT_ENV", "")
    if conda_env != "TLM":
        print(f"⚠️  Warning: Not running in TLM conda environment (current: {conda_env or 'none'})")
        print("   To activate: conda activate TLM")
        response = input("   Continue anyway? (y/N): ")
        if response.lower() != 'y':
            sys.exit(1)
    else:
        print(f"✅ Running in conda environment: {conda_env}")

In [4]:
def setup_environment():
    """Load environment variables and return credentials file path."""
    load_dotenv(dotenv_path="../.env")

    # Sanitize OpenRouter base URL so inline comments are removed
    openrouter_base = os.environ.get("OPENROUTER_API_BASE")
    if openrouter_base:
        cleaned_base = openrouter_base.split("#", 1)[0].strip()
        if cleaned_base != openrouter_base:
            os.environ["OPENROUTER_API_BASE"] = cleaned_base
            print(f"   Sanitized OPENROUTER_API_BASE -> {cleaned_base}")

    print("\n📋 Environment Configuration:")
    summary = {
        "LLM": os.environ.get("LLM", "Not set"),
        "OPENROUTER_API_KEY": "****" if os.environ.get("OPENROUTER_API_KEY") else "Not set",
        "GOOGLE_SHEETS_CREDENTIALS": os.environ.get("GOOGLE_SHEETS_CREDENTIALS", "Not set (using default)"),
        "OPENROUTER_API_BASE": os.environ.get("OPENROUTER_API_BASE", "Default")
    }
    for key, value in summary.items():
        print(f"  {key}: {value}")

    os.environ.pop("OPENAI_API_KEY", None)

    credentials_file = os.path.expanduser(
        os.getenv(
            "GOOGLE_SHEETS_CREDENTIALS",
            "~/Documents/GitHub/nha-sys-sandbox/key/nha-proto-spredsheet-bd3cf7f6603e.json"
        )
    )

    if not os.path.exists(credentials_file):
        print("\n⚠️  Warning: credentials.json not found")
        print(f"Path checked: {credentials_file}")
        print("Please download your service account credentials from Google Cloud Console")
        raise FileNotFoundError(f"Credentials file not found: {credentials_file}")

    print(f"\n✅ Google Sheets credentials found: {credentials_file}")
    return credentials_file

In [5]:
def initialize_gspread(credentials_file):
    """Initialize and return gspread client."""
    try:
        gc = get_gspread_client(credentials_file)
        print("✅ Successfully connected to Google Sheets API")
        print("\nSheet actions available (used by agent):")
        print("  - OpenSpreadsheet: Open a spreadsheet by name/ID")
        print("  - OpenSheet: Switch to a specific worksheet")
        print("  - GetAllValues: Read entire sheet")
        print("  - GetCellValue: Read single cell")
        print("  - UpdateCell: Update single cell")
        print("  - InsertRows: Add new rows")
        print("  - FindCell: Search for values")
        print("  - SortSheetByColumn: Sort data")
        print("  - GetSheetSummary: Get sheet overview")
        return gc
    except Exception as e:
        print(f"❌ Error connecting to Google Sheets: {e}")
        raise

In [6]:
def initialize_llm():

    """Create LLM using configuration from .env file."""

    llm_name = os.getenv("LLM", "openai/gpt-3.5-turbo")

    config_dict = {"llm_name": llm_name, "temperature": 0.0}

    config = LLMConfig(config_dict)

    llm = get_llm_backend(config)



    print(f"✅ LLM initialized: {llm_name}")

    print(f"   Using OPENROUTER_API_KEY from .env file")



    return llm


In [7]:
def create_agent(llm, gspread_client):
    """Create and return SheetManagerAgent instance."""
    return SheetManagerAgent(llm=llm, gspread_client=gspread_client)

In [ ]:
def test_open_spreadsheet(agent, spreadsheet_name="Test Sheet"):
    """Test 1: Open a spreadsheet and get summary."""
    print("=" * 60)
    print("TEST 1: Open spreadsheet and get summary")
    print("=" * 60)

    task = f"Open the spreadsheet named '{spreadsheet_name}' and give me a summary of Sheet1"
    task_pack = TaskPackage(instruction=task)
    response = agent(task_pack)

    print(f"\n📊 Response: {response}\n")
    return response

def test_read_cell(agent):
    """Test 2: Read specific cell values."""
    print("=" * 60)
    print("TEST 2: Read cell values")
    print("=" * 60)

    task = "What is the value in cell B1 of the current sheet?"
    task_pack = TaskPackage(instruction=task)
    response = agent(task_pack)

    print(f"\n📋 Response: {response}\n")
    return response

def test_update_cell(agent):
    """Test 3: Update a cell."""
    print("=" * 60)
    print("TEST 3: Update a cell value")
    print("=" * 60)

    task = "Update cell B2 to contain the text 'Updated by Agent'"
    task_pack = TaskPackage(instruction=task)
    response = agent(task_pack)

    print(f"\n✏️  Response: {response}\n")
    return response

def test_find_cell(agent):
    """Test 4: Find a value."""
    print("=" * 60)
    print("TEST 4: Find a cell containing specific text")
    print("=" * 60)

    task = "Find the cell that contains 'Total' and tell me its location"
    task_pack = TaskPackage(instruction=task)
    response = agent(task_pack)

    print(f"\n🔍 Response: {response}\n")
    return response

def test_complex_workflow(agent):
    """Test 5: Complex workflow - Data Analysis."""
    print("=" * 60)
    print("COMPLEX WORKFLOW: Analyze and update data")
    print("=" * 60)

    task = (
        "In the current sheet, find all cells in column A that contain 'Product',\n"
        "then read the values in column B next to them, and give me a summary."
    )
    task_pack = TaskPackage(instruction=task)
    response = agent(task_pack)

    print(f"\n📊 Response: {response}\n")
    return response

In [ ]:
def main():
    """Main execution function."""
    print("=" * 70)
    print("🚀 Google Sheets Manager Agent Tutorial")
    print("=" * 70)

    verify_environment()

    credentials_file = setup_environment()
    gc = initialize_gspread(credentials_file)
    llm = initialize_llm()
    agent = create_agent(llm, gc)

    print("\n" + "=" * 60)
    print("🎯 Agent initialized and ready!")
    print("=" * 60 + "\n")

    try:
        spreadsheet_id = "1h-F1tMEYXKpm5efWuh0HduXGHsiz9W1alJcMEODvPIY"

        test_open_spreadsheet(agent, spreadsheet_id)
        test_read_cell(agent)
        # test_update_cell(agent)
        # test_find_cell(agent)
        # test_complex_workflow(agent)

        print("\n" + "=" * 60)
        print("✅ All tests completed successfully!")
        print("=" * 60)

    except Exception as e:
        print(f"\n❌ Error during execution: {e}")
        import traceback
        traceback.print_exc()
        raise

In [10]:
# Uncomment to run the full workflow from the notebook.
main()

🚀 Google Sheets Manager Agent Tutorial
✅ Running in conda environment: TLM
   Sanitized OPENROUTER_API_BASE -> https://openrouter.ai/api/v1

📋 Environment Configuration:
  LLM: openai/gpt-3.5-turbo
  OPENROUTER_API_KEY: ****
  GOOGLE_SHEETS_CREDENTIALS: ~/Documents/GitHub/nha-sys-sandbox/key/nha-proto-spredsheet-bd3cf7f6603e.json
  OPENROUTER_API_BASE: https://openrouter.ai/api/v1

✅ Google Sheets credentials found: /Users/ymmtny/Documents/GitHub/nha-sys-sandbox/key/nha-proto-spredsheet-bd3cf7f6603e.json
✅ Successfully connected to Google Sheets API

Sheet actions available (used by agent):
  - OpenSpreadsheet: Open a spreadsheet by name/ID
  - OpenSheet: Switch to a specific worksheet
  - GetAllValues: Read entire sheet
  - GetCellValue: Read single cell
  - UpdateCell: Update single cell
  - InsertRows: Add new rows
  - FindCell: Search for values
  - SortSheetByColumn: Sort data
  - GetSheetSummary: Get sheet overview
✅ LLM initialized: openai/gpt-3.5-turbo
   Using OPENROUTER_API_K